# Notebook 3 — Évaluation du modèle et du pipeline
**Pipeline Big Data de Monitoring de la Désinformation en Temps Réel**  
KOMOSSI Sosso — Master 2 IBDIA, UCAO-UUT 2025-2026

Ce notebook évalue :
1. Les performances sur le jeu de test (F1, AUC, matrice de confusion)
2. La détection de Concept Drift (ADWIN + KSWIN + EDDM)
3. Les métriques temps réel via l'API FastAPI


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, time

MODEL_DIR = '../models/pretrained'
ONNX_DIR  = '../models/onnx'
DATA_DIR  = '../data/processed'
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')
print('Prêt')


## 1. Évaluation sur le jeu de test (F1, AUC, Confusion Matrix)

In [ ]:
from transformers import DistilBertTokenizerFast
from onnxruntime import InferenceSession
from sklearn.metrics import (f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
import torch

test = pd.read_csv(f'{DATA_DIR}/test/test.csv').fillna("")
print(f'Jeu de test : {len(test)} articles')

onnx_file = f'{ONNX_DIR}/model_quantized.onnx'
if not os.path.exists(onnx_file):
    print("ERREUR : modèle ONNX non trouvé — lancer scripts/export_onnx.py")
else:
    tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)
    session   = InferenceSession(onnx_file, providers=["CPUExecutionProvider"])

    preds, probs_list = [], []
    BATCH = 32
    for i in range(0, min(len(test), 2000), BATCH):  # échantillon 2000 pour rapidité
        batch = test.iloc[i:i+BATCH]
        texts = [f"{r['title'][:200]} [SEP] {r['body'][:100]}" for _, r in batch.iterrows()]
        enc   = tokenizer(texts, max_length=128, padding="max_length",
                          truncation=True, return_tensors="np")
        logits = session.run(None, {
            "input_ids":      enc["input_ids"].astype(np.int64),
            "attention_mask": enc["attention_mask"].astype(np.int64)
        })[0]
        p = torch.softmax(torch.tensor(logits), dim=-1).numpy()
        preds.extend(p.argmax(axis=1).tolist())
        probs_list.extend(p[:, 1].tolist())

    y_true = test["label"].values[:len(preds)]
    f1   = f1_score(y_true, preds, average="macro")
    auc  = roc_auc_score(y_true, probs_list)
    print(f"
F1 Score (macro) : {f1:.4f}")
    print(f"AUC-ROC         : {auc:.4f}")
    print()
    print(classification_report(y_true, preds, target_names=["Réel","Fake"]))


## 2. Matrice de confusion + Courbe ROC

In [ ]:
if "preds" in dir() and len(preds) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Matrice de confusion
    cm = confusion_matrix(y_true, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
                xticklabels=["Réel","Fake"], yticklabels=["Réel","Fake"])
    axes[0].set_title("Matrice de Confusion — DistilBERT ONNX INT8")
    axes[0].set_xlabel("Prédiction"); axes[0].set_ylabel("Vérité")

    # Courbe ROC
    fpr, tpr, _ = roc_curve(y_true, probs_list)
    axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC = {auc:.4f}")
    axes[1].plot([0,1],[0,1], color="navy", linestyle="--")
    axes[1].set_xlabel("Taux de faux positifs")
    axes[1].set_ylabel("Taux de vrais positifs")
    axes[1].set_title("Courbe ROC — DistilBERT ONNX INT8")
    axes[1].legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(f"{DATA_DIR}/evaluation_results.png", dpi=150, bbox_inches="tight")
    plt.show()


## 3. Simulation de Concept Drift

In [ ]:
import sys
sys.path.insert(0, "../spark-app/src")
from drift_monitor import DynamicDriftMonitor

monitor = DynamicDriftMonitor()

# Simuler une séquence stable puis un drift
n_stable = 500
n_drift  = 200
np.random.seed(42)
stable_scores = np.random.beta(9, 1, n_stable)      # confiances élevées (stable)
drift_scores  = np.random.beta(1, 1, n_drift)       # confiances aléatoires (drift)
all_scores    = np.concatenate([stable_scores, drift_scores])

composite_history = []
drift_flags       = []
for score in all_scores:
    result = monitor.update(float(score))
    composite_history.append(result["composite_score"])
    drift_flags.append(result["drift"])

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(all_scores, alpha=0.5, color="steelblue", label="Confiance ONNX")
axes[0].axvline(n_stable, color="red", linestyle="--", label="Début drift simulé")
axes[0].set_title("Séquence de confiances du modèle")
axes[0].legend(); axes[0].set_ylim(0, 1)

axes[1].plot(composite_history, color="darkorange", label="Score composite drift")
axes[1].axhline(0.5, color="orange", linestyle="--", label="Seuil détection (0.5)")
axes[1].axhline(0.8, color="red",    linestyle="--", label="Seuil confirmation (0.8)")
axes[1].axvline(n_stable, color="red", linestyle="--")
axes[1].set_title("Score composite ADWIN+KSWIN+EDDM")
axes[1].legend(); axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.savefig(f"{DATA_DIR}/drift_simulation.png", dpi=150, bbox_inches="tight")
plt.show()

n_detected = sum(drift_flags[n_stable:])
print(f"Drift détecté sur {n_detected}/{n_drift} points après le changement")
print(f"Statistiques finales : {json.dumps(monitor.get_stats(), indent=2)}")


## 4. Métriques temps réel via l'API

In [ ]:
import requests
API_URL = "http://localhost:8000"
try:
    h = requests.get(f"{API_URL}/health", timeout=3)
    print(f"Santé API : {h.json()}")
    s = requests.get(f"{API_URL}/api/v1/stats", timeout=3)
    stats = s.json()
    print()
    print(f"Total articles  : {stats.get('total_articles',0):,}")
    print(f"Articles fake   : {stats.get('fake_articles',0):,}")
    print(f"Taux fake       : {stats.get('fake_rate',0):.1f}%")
    print(f"Événements drift: {stats.get('drift_events',0)}")
    print(f"Articles/heure  : {stats.get('articles_last_hour',0):,}")
except Exception as e:
    print(f"API non disponible : {e}")
    print("Lancer d'abord : docker compose up -d")
